In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from flask import Flask, request, jsonify

app = Flask(__name__)

# Load model once
model = YOLO("model/runs/train/toy_animals_full/weights/best.pt")

GRID_ROWS = 3
GRID_COLS = 2

def detect_animal(target_label):
    """Detect specific animal and return its grid location."""
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        return {"status": "error", "message": "Camera not found"}

    for _ in range(15):  # Try multiple frames
        ret, frame = cap.read()
        if not ret:
            continue

        h, w = frame.shape[:2]
        results = model(frame)[0]

        for box in results.boxes:
            cls = int(box.cls[0])
            label = model.names[cls]

            # Only return the target animal
            if label.lower() == target_label.lower():
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                cx = int((x1 + x2) / 2)
                cy = int((y1 + y2) / 2)

                # Compute grid cell
                cell_w = w // GRID_COLS
                cell_h = h // GRID_ROWS
                row = cy // cell_h
                col = cx // cell_w

                cap.release()
                return {
                    "status": "success",
                    "animal": label,
                    "cx": cx,
                    "cy": cy,
                    "grid": [int(row), int(col)]
                }

    cap.release()
    return {"status": "not_found", "message": f"{target_label} not found"}

@app.route("/find", methods=["POST"])
def find():
    data = request.json
    animal = data.get("animal", "")
    print(f"Received command to find: {animal}")
    result = detect_animal(animal)
    return jsonify(result)

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5050)
